In [10]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

Libraries imported!


In [11]:
df = pd.read_csv('products_all_fixed.csv')

print(f"Total rows     : {len(df)}")
print(f"Columns        : {list(df.columns)}")
print(f"Categories     : {list(df['category'].unique())}")
df.head(3)

Total rows     : 58000
Columns        : ['product_id', 'product_name', 'brand', 'category', 'subcategory', 'seller', 'price_NPR', 'currency']
Categories     : ['Personal Care', 'Food & Beverages', 'Cosmetics', 'Household', 'Sports']


,product_id,product_name,brand,category,subcategory,seller,price_NPR,currency
0,P0001,Dove Shampoo 500ml,Dove,Personal Care,Shampoo,SastoDeal,770.86,NPR
1,P0002,Dove Shampoo 500ml,Dove,Personal Care,Shampoo,Daraz Nepal,911.60,NPR
2,P0003,Dove Shampoo 500ml,Dove,Personal Care,Shampoo,Amazon,658.87,NPR


In [12]:
# Combine columns as input text
df['text'] = df['product_name'] + ' ' + df['brand'] + ' ' + df['subcategory']

# One row per product — remove duplicates
df_unique = df.drop_duplicates(subset='product_name')

X = df_unique['text']       # Input  → text
y = df_unique['category']   # Output → category label

print(f"Unique products  : {len(df_unique)}")
print(f"Sample input     : {X.iloc[0]}")
print(f"Sample label     : {y.iloc[0]}")

Unique products  : 237
Sample input     : Dove Shampoo 500ml Dove Shampoo
Sample label     : Personal Care


The Function\
pythontrain_test_split(X, y, test_size=0.2, random_state=42)\
It takes your data and splits it into 2 parts automatically.\
\
Parameter--------       Value-------------           Meaning\
X-input----------         all   product----   texts to split\
y-output-----------        label all -------       category labels to split\
test_size=0.2------   0.2 = 20% ------------       20% goes to testing, 80% to training\
random_state=42--- any fixed number --------ensures same split every time you run\

What comes out — 4 variables\
X_train → 80% of product texts     (model learns FROM these)\
X_test  → 20% of product texts     (model is tested ON these)\
y_train → 80% of category labels   (answers for training)\
y_test  → 20% of category labels   (answers for testing)\

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 189
Testing samples  : 48


In [14]:
#  Convert Text to Numbers (TF-IDF)
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f"TF-IDF vectorization done!")
print(f"Vocabulary size  : {len(vectorizer.vocabulary_)}")
print(f"Matrix shape     : {X_train_vec.shape}")

TF-IDF vectorization done!
Vocabulary size  : 389
Matrix shape     : (189, 389)


Train the ML Model\
This is where actual machine learning happens.\
We use Multinomial Naive Bayes — a simple but powerful algorithm for text classification.\
\
Words like "shampoo", "soap", "lotion"   → probably Personal Care\
Words like "noodles", "cola", "biscuit"  → probably Food & Beverages\
Words like "lipstick", "mascara"         → probably Cosmetics\
Words like "detergent", "dishwash"       → probably Household\
Words like "protein", "gatorade"         → probably Sports\
The model studies all 153 training products and remembers these patterns.

In [15]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("Multinomial Naive Bayes model trained!")

Multinomial Naive Bayes model trained!


In [16]:
# Evaluate the Model
# We test the model on the 39 unseen products and see how well it predicts the category.
y_pred = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f"Model Accuracy : {accuracy:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

Model Accuracy : 75.0%

Detailed Report:
                  precision    recall  f1-score   support

       Cosmetics       1.00      0.42      0.59        12
Food & Beverages       0.68      1.00      0.81        13
       Household       0.75      1.00      0.86         3
   Personal Care       0.71      1.00      0.83        12
          Sports       1.00      0.38      0.55         8

        accuracy                           0.75        48
       macro avg       0.83      0.76      0.73        48
    weighted avg       0.83      0.75      0.72        48



In [17]:
# Save Model
# model.pkl       → the trained ML brain
# vectorizer.pkl  → the text-to-number converter

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("model.pkl saved!")
print("vectorizer.pkl saved!")
print("\nTraining complete! Ready for app.py")

model.pkl saved!
vectorizer.pkl saved!

Training complete! Ready for app.py
